### Web Page Classifier Module

Classifies web pages into predefined categories based on text content using sentence embeddings and centroid-based similarity.

---

#### Pipeline Steps

##### Step 1 — Scrape & Store HTML Source (`scrape_page.py`)
Takes an Excel sheet and source name as input, scrapes each URL, and saves the output as a new Excel file with an added `html_source` column containing the raw HTML of each page.

##### Step 2 — Generate Embeddings
Loads the scraped Excel, extracts visible text from `html_source`, and encodes each entry using a sentence transformer model. Saves all embeddings along with their labels to a `.pkl` file.

##### Step 3 — Build Class Centroids
Loads the embeddings, groups them by class label, and computes a centroid vector for each class by averaging all embeddings in that group. Saves the centroid map to a `.pkl` file.

##### Step 4 — Classify New Web Pages
Given a new URL, scrapes its content, generates an embedding, and computes cosine similarity against all stored centroids. The class with the highest similarity score is assigned as the predicted label.

In [1]:
# Read dataframe , the dataframe contains a column named "html_source" which contains the HTML source code of the scraped web pages
# Create 'clean_text' column by converting HTML to plain text
# Create 'embedding' column by generating sentence embeddings from the 'clean_text' column using a pre-trained model from the sentence-transformers library

from bs4 import BeautifulSoup
import re
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np


def html_to_text(html: str) -> str:
    if not html:
        return ""

    soup = BeautifulSoup(html, "lxml")

    # Remove non-content elements
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def create_html_embeddings(
    df: pd.DataFrame,
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    text_column: str = "html_source",
    embedding_column: str = "embedding"
) -> pd.DataFrame:
    """
    Create embeddings from HTML source and store them in DataFrame.
    """

    if text_column not in df.columns:
        raise ValueError(f"'{text_column}' column not found")

    df = df.copy()

    # Convert HTML → text
    df["clean_text"] = df[text_column].apply(html_to_text)

    model = SentenceTransformer(model_name)

    texts = df["clean_text"].tolist()
    embeddings = model.encode(
        texts,
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True  # important for cosine similarity
    )

    df[embedding_column] = list(embeddings)

    return df


c:\Users\Yaseen T P\Desktop\Gordian Redis system\gorenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
source = 'ductstore.co'
df = pd.read_excel(f"{source}_scraped.xlsx")
embeddings_df = create_html_embeddings(df)
embeddings_df.to_pickle(f"{source}_embeddings.pkl")  

Batches: 100%|██████████| 7/7 [00:09<00:00,  1.32s/it]


#### Create centroids and predict type

In [6]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

from sklearn.model_selection import train_test_split

def split_data(df, test_size=0.2, random_state=42, type_col="Type", min_samples=2):
    counts = df[type_col].value_counts()
    
    rare = df[df[type_col].isin(counts[counts < min_samples].index)]
    common = df[df[type_col].isin(counts[counts >= min_samples].index)]

    train_common, eval_df = train_test_split(
        common,
        test_size=test_size,
        random_state=random_state,
        stratify=common[type_col]
    )

    train_df = pd.concat([train_common, rare]).reset_index(drop=True)

    return train_df, eval_df.reset_index(drop=True)

def build_centroids(df, type_col="Type", emb_col="embedding"):
    """
    Returns a dict: {type: mean_embedding}
    """
    prototypes = {}

    for t in sorted(df[type_col].unique()):
        class_embeddings = np.array(df[df[type_col] == t][emb_col].tolist())
        prototypes[t] = class_embeddings.mean(axis=0)

    return prototypes


# Func for predict the type of a page based on its embedding and the centroids of each type, using cosine similarity

def predict_type(embedding, centroids, threshold=0.5):
    best_type = None
    best_score = -1

    emb = np.array(embedding).reshape(1, -1)

    for t, centroid_emb in centroids.items():
        centroid_emb = np.array(centroid_emb).reshape(1, -1)
        score = cosine_similarity(emb, centroid_emb)[0][0]

        if score > best_score:
            best_score = score
            best_type = t

    if best_score < threshold:
        return "unknown", best_score

    return best_type, best_score


def predict(data, centroids, type_col="Type", emb_col="embedding", threshold=0.5):
    """
    Accepts a DataFrame or a single row (dict/Series),
    returns a DataFrame with predicted_type and similarity_score columns.
    """
    if isinstance(data, (dict, pd.Series)):
        data = pd.DataFrame([data])

    df = data.copy()

    preds, scores = [], []

    for _, row in df.iterrows():
        pred_type, score = predict_type(row[emb_col], centroids, threshold)
        preds.append(pred_type)
        scores.append(score)

    df["predicted_type"] = preds
    df["similarity_score"] = scores

    return df


#### Training

In [7]:
df = pd.read_pickle(f"{source}_embeddings.pkl") 
train_df, eval_df = split_data(df)
centroids = build_centroids(train_df)


#### Prediction

In [8]:
predictions_df = predict(eval_df, centroids)

#### Evaluation

In [9]:



def evaluate_type_classifier(
    df,
    prototypes,
    type_col="Type",
    emb_col="embedding"
):
    df = df.copy()

    preds = []
    scores = []

    for _, row in df.iterrows():
        pred_type, score = predict_type(row[emb_col], prototypes)
        preds.append(pred_type)
        scores.append(score)

    df["predicted_type"] = preds
    df["similarity_score"] = scores
    df["correct"] = df["predicted_type"] == df[type_col]

    accuracy = df["correct"].mean()

    return df, accuracy


In [10]:
results_df, accuracy = evaluate_type_classifier(eval_df, centroids)

print(f"Accuracy: {accuracy:.2%}")


Accuracy: 100.00%
